# فسّرها لي — تحليل الشعر العربي بالذكاء الاصطناعي

ميزة متكاملة تجمع ثلاثة نماذج:
- **بحر الشعر** — نموذج تصنيف الأوزان العروضية
- **العصر الزمني** — نموذج BERTerav2 (كلاسيكي / حديث)
- **موضوع القصيدة** — نموذج LSeN (7 موضوعات)
- **التفسير الأدبي** — Claude AI بمعايير نقدية متخصصة


## Step 1 — Install Required Libraries

We install the core libraries needed across all three models and the OpenAI client:
- `transformers` — loads pretrained BERT-based models for meter, era, and topic
- `accelerate` + `safetensors` — efficient model loading from Hugging Face format
- `torch` — runs inference on GPU/CPU
- `openai` — connects to GPT-4o for the literary explanation step
- `datasets` — used if you need to reload training data for debugging

In [ ]:
!pip install -q transformers accelerate safetensors torch openai datasets

## Step 2 — Import Libraries

We import only what is strictly needed:
- `re`, `html` — Arabic text cleaning (remove diacritics, HTML tags)
- `pickle` — load the topic model's label mapping saved during training
- `torch` + `numpy` — tensor operations and probability sorting
- `Counter` — majority voting for meter prediction across verses
- `AutoTokenizer` / `AutoModelForSequenceClassification` — generic HuggingFace interface that works for all three models without model-specific code

In [ ]:
import os, re, pickle, html
import numpy as np
import torch
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import openai

## Step 3 — Mount Google Drive

All three trained models are stored on Google Drive.
Mounting Drive gives Colab direct file-system access to the model weights without downloading them again.

> If you're running this outside Colab (e.g. on a server), replace the Drive paths with absolute local paths.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 4 — Load the Meter Classification Model

**Path:** `/content/drive/MyDrive/Model`

This model was fine-tuned to classify the **prosodic meter** (البحر الشعري) of Arabic poetry.
It covers all 14 classical Arabic meters (e.g. Taweel, Baseet, Kamil).

**Why verse-by-verse prediction?**
Arabic meters are defined at the single-verse level, not the full poem.
We predict each verse independently, then use **majority voting** across all verses to determine the dominant meter of the poem.

In [8]:
METER_MODEL_PATH = "/content/drive/MyDrive/Model"

meter_tokenizer = AutoTokenizer.from_pretrained(METER_MODEL_PATH)
meter_model     = AutoModelForSequenceClassification.from_pretrained(METER_MODEL_PATH)
meter_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
meter_model.to(device)

labels_meter = ['saree', 'kamel', 'mutakareb', 'mutadarak', 'munsareh',
                'madeed', 'mujtath', 'ramal', 'baseet', 'khafeef',
                'taweel', 'wafer', 'hazaj', 'rajaz']

meter_arabic = {
    'saree': 'السريع',    'kamel': 'الكامل',       'mutakareb': 'المتقارب',
    'mutadarak': 'المتدارك', 'munsareh': 'المنسرح', 'madeed': 'المديد',
    'mujtath': 'المجتث',  'ramal': 'الرمل',         'baseet': 'البسيط',
    'khafeef': 'الخفيف',  'taweel': 'الطويل',       'wafer': 'الوافر',
    'hazaj': 'الهزج',     'rajaz': 'الرجز'
}

print(" نموذج البحر الشعري جاهز")
print(f"  الجهاز: {device}")

Loading weights:   0%|          | 0/169 [00:00<?, ?it/s]

 نموذج البحر الشعري جاهز
  الجهاز: cuda


## Step 5 — Load the Era Classification Model (BERTerav2)

**Path:** `/content/drive/MyDrive/Model/Time_period_classification_Model_V2/BERTerav2`

This model is a fine-tuned AraBERT that classifies whether a poem belongs to the **Classical** (قديم) or **Modern** (حديث) era.

**Why does era matter for analysis?**
Classical poems follow strict meter and rhyme rules and carry archaic vocabulary.
Modern poems often break meter deliberately for emotional effect.
Knowing the era helps the LLM frame its explanation correctly.

In [10]:
ERA_MODEL_DIR = "/content/drive/MyDrive/Model/Time_period_classification_Model_V2/BERTerav2"

era_tokenizer = AutoTokenizer.from_pretrained(ERA_MODEL_DIR)
era_model     = AutoModelForSequenceClassification.from_pretrained(ERA_MODEL_DIR)
era_model.to(device)
era_model.eval()

print(" نموذج العصر الزمني جاهز")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

 نموذج العصر الزمني جاهز


## Step 6 — Load the Topic Classification Model (LSeN)

**Path:** `/content/drive/MyDrive/Model/Topic_classifiication/best_model`

This model classifies the poem into one of **7 thematic categories**:
- غزل رومانسي (Romance)
- مدح (Praise)
- رثاء (Elegy)
- دينية (Religious)
- وجداني / عاطفة وحنين (Emotional)
- وطنية (Patriotic)
- هجاء وذم (Satire)

The `label_info.pkl` file maps numeric model outputs back to Arabic label names.
It was saved during training and must be loaded alongside the model weights.

In [14]:
import os
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for f in files:
        if "label_info" in f or "best_model" in f.lower():
            print(os.path.join(root, f))

/content/drive/MyDrive/Model/Topic_classifiication/best_model-20260416T175330Z-3-001.zip
/content/drive/MyDrive/Model/Topic_classifiication/best_model/label_info.pkl


In [15]:
TOPIC_MODEL_PATH  = "/content/drive/MyDrive/Model/Topic_classifiication/best_model"
TOPIC_LABELS_PATH = "/content/drive/MyDrive/Model/Topic_classifiication/best_model/label_info.pkl"

topic_tokenizer = AutoTokenizer.from_pretrained(TOPIC_MODEL_PATH)
topic_model     = AutoModelForSequenceClassification.from_pretrained(TOPIC_MODEL_PATH)
topic_model.to(device)
topic_model.eval()

with open(TOPIC_LABELS_PATH, "rb") as f:
    label_info = pickle.load(f)

id2label_topic = label_info["id2label"]

print(" نموذج الموضوع جاهز")
print(f"  الموضوعات: {list(id2label_topic.values())}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

 نموذج الموضوع جاهز
  الموضوعات: ['دينية', 'رثاء', 'غزل_رومانسي', 'مدح', 'هجاء_ذم', 'وجداني', 'وطنية']


## Step 7 — Preprocessing and Classification Functions

This cell defines four functions that power the entire feature:

| Function | Input | Output |
|---|---|---|
| `clean_arabic()` | raw poem string | normalized Arabic text |
| `predict_meter()` | poem string | meter name + confidence |
| `predict_era()` | poem string | era label + probabilities |
| `predict_topic()` | poem string | topic label + top-3 ranking |

**Why do we normalize Arabic text?**
Arabic script has optional diacritics (تشكيل), elongation characters (كشيدة), and multiple forms of the same letter (أ إ آ ا).
Our models were trained on normalized text, so we must apply the same normalization at inference time to avoid mismatches.

In [ ]:
ARABIC_DIACRITICS = re.compile(r'[\u0617-\u061A\u064B-\u0652\u0670\u06D6-\u06ED]')

def clean_arabic(text):
    """Normalize Arabic text to match training-time preprocessing."""
    if not text:
        return ""
    text = html.unescape(str(text))
    text = re.sub(r"<.*?>", " ", text)       # strip HTML tags
    text = text.replace("\u0640", "")        # remove kashida (tatweel)
    text = ARABIC_DIACRITICS.sub("", text)   # remove all diacritics
    text = re.sub(r'[\u0623\u0625\u0622\u0671]', '\u0627', text)  # unify alef variants -> ا
    text = text.replace("\u0629", "\u0647")  # taa marbuta -> ha
    text = re.sub(r"[0-9\u0660-\u0669]", " ", text)  # remove digits (Arabic + Western)
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text) # keep Arabic chars only
    text = re.sub(r"\s+", " ", text).strip()
    return text


def predict_meter(poem_text: str) -> dict:
    """
    Predict the prosodic meter of a poem using majority voting.
    Each verse is classified independently; the most frequent prediction wins.
    """
    verses = [v.replace("#", " ").strip() for v in poem_text.strip().split("\n") if v.strip()]
    predictions = []
    for verse in verses:
        inputs = meter_tokenizer(verse, return_tensors="pt", truncation=True,
                                 max_length=32, padding="max_length")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            probs = torch.softmax(meter_model(**inputs).logits, dim=-1)[0]
        pred_id    = torch.argmax(probs).item()
        confidence = probs[pred_id].item()
        predictions.append((labels_meter[pred_id], confidence))

    votes     = [p[0] for p in predictions]
    top_meter = Counter(votes).most_common(1)[0][0]
    avg_conf  = sum(c for _, c in predictions) / len(predictions)
    return {
        "meter_ar":   meter_arabic[top_meter],
        "meter_en":   top_meter,
        "confidence": round(avg_conf, 3),
    }


def predict_era(poem_text: str) -> dict:
    """
    Classify the poem as Classical (قديم) or Modern (حديث).
    Uses BERTerav2 fine-tuned on a labeled Arabic poetry corpus.
    """
    cleaned = clean_arabic(poem_text)
    encoded = era_tokenizer(cleaned, padding="max_length", truncation=True,
                            max_length=256, return_tensors="pt")
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        probs = torch.softmax(era_model(**encoded).logits, dim=-1).cpu().numpy()[0]
    label_names = ["قديم", "حديث"]
    pred_idx    = int(np.argmax(probs))
    return {
        "era":                   label_names[pred_idx],
        "classical_probability": round(float(probs[0]), 4),
        "modern_probability":    round(float(probs[1]), 4),
    }


def predict_topic(poem_text: str) -> dict:
    """
    Classify the poem into one of 7 thematic categories.
    Returns the top prediction plus a ranked top-3 list.
    """
    cleaned = clean_arabic(poem_text)
    inputs  = topic_tokenizer(cleaned, truncation=True, max_length=512,
                              return_tensors="pt", padding=True)
    inputs  = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        probs = torch.softmax(topic_model(**inputs).logits, dim=-1)[0].cpu().numpy()
    top3 = np.argsort(probs)[::-1][:3]
    return {
        "topic":      id2label_topic[int(top3[0])],
        "confidence": round(float(probs[top3[0]]), 3),
        "top3": [
            {"label": id2label_topic[int(i)], "prob": round(float(probs[i]), 3)}
            for i in top3
        ],
    }

print("✓ All classification functions ready")

## Step 8 — Set Up OpenAI Client

We use **GPT-4o** as the literary analysis engine.

**Why GPT-4o and not a smaller model?**
Literary analysis of Arabic poetry requires:
- Deep understanding of classical Arabic vocabulary
- Awareness of rhetorical devices (بلاغة)
- Ability to connect meter, theme, and emotion coherently

GPT-4o is currently the strongest model for nuanced Arabic literary tasks.

> **Security note:** Never commit your API key to GitHub.
> Use environment variables or a `.env` file in production.

In [ ]:
import os
import openai

# ← Paste your OpenAI API key here (never share or commit this key)
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY_HERE"

openai_client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("✓ OpenAI GPT-4o ready")

## Step 9 — Main Analysis Function: `fasserha_li`

This is the core function of the entire feature. It:

1. Runs all three classification models on the input poem
2. Injects the results into a carefully engineered prompt
3. Calls GPT-4o to generate a full literary analysis
4. Returns a structured dict used by both the print function and the API

**Why inject model results into the prompt?**
GPT-4o alone cannot reliably identify the exact Arabic meter or fine-grained topic category — those require specialized fine-tuned models.
By passing the model outputs as *grounded facts*, we get an analysis that is:
- Accurate on technical details (meter name, era, theme)
- Rich and fluent in the literary explanation

**Prompt design — inspired by real Arabic poetry analysis platforms:**
The prompt mirrors the structure used by sites like الديوان and أدب, which organize poem analysis into:
- المعنى الإجمالي (overall meaning)
- شرح المفردات (vocabulary explanation)
- الصور البلاغية (rhetorical images)
- الموسيقى الشعرية (poetic music / meter feel)

In [ ]:
TOPIC_AR = {
    "غزل_رومانسي": "غزل ورومانسية",
    "هجاء_ذم":     "هجاء وذم",
    "وجداني":      "عاطفة وحنين",
    "مدح":         "مدح وإطراء",
    "رثاء":        "رثاء وحزن",
    "دينية":       "شعر ديني",
    "وطنية":       "شعر وطني",
}


def fasserha_li(poem_text: str) -> dict:
    """
    Main function for the Fasserha Li feature.
    Combines three classification models with GPT-4o literary analysis.

    Args:
        poem_text: Raw Arabic poem (one verse or multiple lines)

    Returns:
        dict with keys: poem, meter, era, topic, explanation
    """

    # Step 1 — Run the three classification models
    meter_result = predict_meter(poem_text)
    era_result   = predict_era(poem_text)
    topic_result = predict_topic(poem_text)

    meter_name = meter_result["meter_ar"]
    era_name   = era_result["era"]
    topic_name = TOPIC_AR.get(topic_result["topic"], topic_result["topic"])

    # Step 2 — System role: establishes GPT-4o as a specialized Arabic literary critic
    system_prompt = """أنت ناقد أدبي عربي متخصص، تُحلِّل الشعر العربي بأسلوب المواقع الأدبية الاحترافية.
تتبع في تحليلك المنهج المستخدم في كبرى منصات الشعر العربي:
- تبدأ بالمعنى الإجمالي للنص
- تشرح المفردات الصعبة أو الموحية
- تكشف الصور البلاغية وجمالها
- توضح الموسيقى الشعرية وأثرها العاطفي

قواعد ثابتة لا تحيد عنها:
- اكتب بعربية فصيحة واضحة، بعيدة عن الحشو والتعقيد
- اقتبس من النص مباشرة عند كل نقطة تحليلية
- لا تذكر أرقاماً أو نسباً مئوية
- إذا كان النص بيتاً واحداً، عمّق التحليل بدل أن توسّعه"""

    # Step 3 — User prompt: structured like real Arabic poetry analysis websites
    user_prompt = f"""## النص الشعري:
{poem_text}

## المعطيات المُستخرجة:
- البحر: {meter_name}
- العصر: {era_name}
- الموضوع: {topic_name}

---

## التحليل الأدبي المطلوب:

### أولاً — المعنى الإجمالي
في جملتين أو ثلاث، ما الذي يقوله هذا النص في مجمله؟ وما الحالة الشعورية التي يعبّر عنها؟

### ثانياً — شرح المفردات والأساليب
- اشرح الكلمات أو التراكيب التي تحتاج توضيحاً (بصيغة: الكلمة ← معناها)
- إن كان النص حديثاً وألفاظه واضحة، انتقل مباشرة للصور البلاغية

### ثالثاً — الصور البلاغية
اذكر تحديداً (باقتباس من النص) صورة بيانية واحدة على الأقل:
- التشبيه، الاستعارة، الكناية، أو أي أسلوب بلاغي لافت
- اشرح لماذا هذه الصورة بالذات قوية أو جميلة

### رابعاً — الموسيقى الشعرية
بحر {meter_name}: 
- صفه في كلمة واحدة (هادئ / متدفق / صاخب / رزين / سريع...)
- كيف يخدم هذا الإيقاع المشاعر التي يحملها النص؟

### خامساً — خلاصة ناقد
جملة أو جملتان تلتقطان ما يميّز هذا النص أو ما يجعله يستحق القراءة.

---
اجعل تحليلك متدفقاً ومتصلاً، كما لو كنت تكتب مقالاً أدبياً قصيراً لقارئ مثقف يحب الشعر لكنه ليس متخصصاً."""

    # Step 4 — Call GPT-4o
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        max_tokens=1400,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ]
    )

    explanation = response.choices[0].message.content

    # Step 5 — Return structured result
    return {
        "poem":        poem_text,
        "meter":       meter_result,
        "era":         era_result,
        "topic":       {**topic_result, "topic": topic_name},
        "explanation": explanation,
    }


def print_result(result: dict):
    """Pretty-print the analysis result for notebook use."""
    sep = "─" * 50
    print(f"\n{sep}")
    print(f"  البحر  :  {result['meter']['meter_ar']}")
    print(f"  العصر  :  {result['era']['era']}")
    print(f"  الموضوع:  {result['topic']['topic']}")
    print(f"{sep}")
    print(f"\n{result['explanation']}")
    print(f"\n{sep}")

print("✓ fasserha_li ready")

## Step 10 — Test with a Sample Poem

We test with امرؤ القيس's Mu'allaqah — one of the most famous classical Arabic poems.
Expected output: meter = الطويل, era = قديم, topic = غزل أو وجداني.

In [39]:
poem = """
قفا نبك من ذكرى حبيب ومنزل
بسقط اللوى بين الدخول فحومل
فتوضح فالمقراة لم يعف رسمها
لما نسجتها من جنوب وشمأل
"""

result = fasserha_li(poem)
print_result(result)

نتائج النماذج:
 البحر : الطويل (99.1%)
  العصر : قديم
  الموضوع: وجداني (61.4%)

 التفسير الأدبي:

### أولاً — الفكرة العامة

يتحدث النص عن ذكرى حب ومكان مميز في حياة الشاعر، حيث يتذكر اللحظات والأماكن التي قضى فيها وقتاً مع حبيبه. يشير النص إلى مشاعر الحنين والشوق للماضي، ممزوجة بنبرة من الأسى والحزن على مرور الزمن واندثار الآثار.

### ثانياً — المعنى التفصيلي

- يبدأ النص بدعوة الشاعر لنفسه وربما لشخص آخر للتوقف عند الذكريات اللطيفة التي عاشها مع حبيبه، في مكان محدد يدعى "سقط اللوى" بين معالم جغرافية معينة ("الدخول" و"فحومل").
- ينتقل الشاعر لوصف الحالة المُحتضرة لهذا المكان، حيث لا تزال آثاره باقية رغم عوامل الطبيعة التي عملت على محو ملامحه ("لم يعف رسمها لما نسجتها من جنوب وشمال").

يظهر في النص تتابع للأحداث من الدعوة للذكرى إلى وصف أثار الزمن على تلك الأماكن، مما يعكس التطور الطبيعي للحنين إلى المواجهة مع مرور الزمن.

### ثالثاً — الجمال الأدبي

- **البلاغة:** يظهر في النص الاستخدام الجميل للطباق في قوله "جنوب وشمأل"، الذي يضفي تناقضاً يجذب الانتباه ويمثّل اختلاف الظروف المناخية 

## Step 11 — Manual Input

Run this cell to analyze any poem interactively.
Paste the poem text when prompted and press Enter.

In [ ]:
poem_input = input("أدخلي القصيدة أو الأبيات (Enter مرتين للإنهاء):\n")
result = fasserha_li(poem_input)
print_result(result)

## Step 12 — API-Ready Function for FastAPI + Render + Lovable

This cell wraps `fasserha_li` into a clean JSON response
that maps directly to what the Lovable frontend expects.

In [ ]:
def fasserha_api_response(poem_text: str) -> dict:
    """
    مخرجات جاهزة للـ API — تُرسل مباشرة لـ Lovable كـ JSON
    """
    result = fasserha_li(poem_text)
    return {
        "success": True,
        "data": {
            "meter": {
                "arabic":     result["meter"]["meter_ar"],
                "english":    result["meter"]["meter_en"],
                "confidence": result["meter"]["confidence"],
            },
            "era": {
                "label":              result["era"]["era"],
                "classical_prob":     result["era"]["classical_probability"],
                "modern_prob":        result["era"]["modern_probability"],
            },
            "topic": {
                "label":      result["topic"]["topic"],
                "confidence": result["topic"]["confidence"],
                "top3":       result["topic"]["top3"],
            },
            "explanation": result["explanation"],
        }
    }


# ← اختبار سريع
test_poem = "يا قصيدة عمري اللي تكتبيني\nكل يوم حرف من نغمة هواك"
api_response = fasserha_api_response(test_poem)
print("✓ API Response Sample:")
print(f"  meter:       {api_response['data']['meter']['arabic']}")
print(f"  era:         {api_response['data']['era']['label']}")
print(f"  topic:       {api_response['data']['topic']['label']}")
print(f"  explanation: {api_response['data']['explanation'][:100]}...")

---

## Deployment Guide — Connecting to FastAPI → Render → Lovable

This section is written for teammates who will handle the backend and frontend integration.

---

### Architecture Overview

```
Lovable (Frontend)
      ↓  POST /fasserha  { "poem": "..." }
FastAPI (Backend on Render)
      ↓  loads models from Hugging Face or Drive
fasserha_li() runs → returns JSON
      ↓
Lovable receives { meter, era, topic, explanation }
      and renders the analysis UI
```

---

### Step A — Convert this notebook into a FastAPI app

Create a file called `main.py`:

```python
from fastapi import FastAPI
from pydantic import BaseModel
from fasserha_module import fasserha_api_response  # extract functions from this notebook

app = FastAPI()

class PoemRequest(BaseModel):
    poem: str

@app.post("/fasserha")
def analyze(req: PoemRequest):
    return fasserha_api_response(req.poem)
```

**How to extract the functions:**
Copy these from this notebook into a file called `fasserha_module.py`:
- `clean_arabic()`
- `predict_meter()`
- `predict_era()`
- `predict_topic()`
- `fasserha_li()`
- `fasserha_api_response()`

Also copy the model loading code and run it once at startup (outside any function).

---

### Step B — Create `requirements.txt`

```
fastapi
uvicorn
transformers
torch
openai
safetensors
accelerate
numpy
```

---

### Step C — Deploy on Render

1. Push your project to a GitHub repo with this structure:
```
project/
├── main.py
├── fasserha_module.py
├── requirements.txt
└── models/          ← model weights (or load from HF Hub)
```

2. Go to [render.com](https://render.com) → New → Web Service → connect your repo

3. Set these in Render's dashboard:
   - **Build command:** `pip install -r requirements.txt`
   - **Start command:** `uvicorn main:app --host 0.0.0.0 --port $PORT`
   - **Environment variable:** `OPENAI_API_KEY` = your key

4. Render gives you a public URL like:
   `https://your-app.onrender.com`

> **Important:** The free Render tier sleeps after 15 min of inactivity.
> First request after sleep takes ~30s while the server wakes up.
> Upgrade to a paid instance for production use.

---

### Step D — Connect Lovable to the API

In Lovable, in the component that handles the poem input, add a fetch call:

```javascript
const analyzePoem = async (poemText) => {
  const response = await fetch("https://your-app.onrender.com/fasserha", {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ poem: poemText })
  });
  const data = await response.json();
  // data.data.meter.arabic   → البحر
  // data.data.era.label      → العصر
  // data.data.topic.label    → الموضوع
  // data.data.explanation    → التفسير الكامل
  return data;
};
```

---

### Expected JSON Response Shape

```json
{
  "success": true,
  "data": {
    "meter":  { "arabic": "الطويل", "english": "taweel", "confidence": 0.94 },
    "era":    { "label": "قديم", "classical_prob": 0.97, "modern_prob": 0.03 },
    "topic":  { "label": "غزل ورومانسية", "confidence": 0.81, "top3": [...] },
    "explanation": "..."
  }
}
```

---

### Notes for teammates

- **Model loading is slow** (~30s on first request). Load all models once at server startup, not inside the endpoint function.
- **GPU on Render:** Render free tier is CPU-only. For faster inference, consider Modal.com (GPU serverless) or HuggingFace Spaces with a GPU instance.
- **CORS:** Add `fastapi.middleware.cors` if Lovable and the API are on different domains (they will be).
- **Rate limiting:** OpenAI charges per token. Add basic rate limiting if the feature will be public-facing.
